In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# 자신의 프로젝트 폴더 경로로 변경하세요
from pathlib import Path

my_path = Path('/content/drive/MyDrive/비트메이트_TP01')

In [3]:
# =========================
# 1) 설치 + 모델 로드
# =========================

!pip install -q -U openai-whisper transformers accelerate bitsandbytes sentencepiece qwen-tts soundfile gradio
!apt-get -qq update
!apt-get -qq install -y ffmpeg sox libsox-fmt-all

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 50.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [8]:
# =========================
# 1) 모델 로드
# =========================
import os
import gc
from pathlib import Path

import torch
import whisper
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from qwen_tts import Qwen3TTSModel

# Whisper 로드
whisper_model = whisper.load_model("turbo")
print("Whisper 로드 완료")

# Qwen 로드
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Qwen 로드 완료")

# TTS 로드
speaker = "jhc100"
ckpt_root = my_path / "qwen3_ft_output" / speaker

epochs = []
for name in os.listdir(ckpt_root):
    if name.startswith("checkpoint-epoch-"):
        try:
            epochs.append(int(name.split("-")[-1]))
        except ValueError:
            pass

if not epochs:
    raise RuntimeError(f"checkpoint-epoch-* 폴더를 찾지 못했습니다: {ckpt_root}")

selected_epoch = max(epochs)
target_ckpt = ckpt_root / f"checkpoint-epoch-{selected_epoch}"

tts_model = Qwen3TTSModel.from_pretrained(
    str(target_ckpt),
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

print("TTS 로드 완료")
print("모든 모델 로드 완료")

Whisper 로드 완료


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen 로드 완료
TTS 로드 완료
모든 모델 로드 완료


In [9]:
# =========================
# 2) 함수
# =========================

def chat_qwen(user_text, system_prompt="당신은 딸에게 대답하는 아버지 입니다."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(qwen_model.device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return response


def record_audio(sec=10, filename='/content/recorded_audio.wav'):
  print(f"🎤 {sec}초 동안 녹음을 시작합니다...")

  # JS 실행
  display(Javascript(RECORD_JS))

  # JS 함수 호출 및 데이터 수신
  s = output.eval_js(f'record({sec * 1000})')

  # Base64 디코딩
  b = b64decode(s.split(',')[1])

  # 파일 저장
  with open(filename, 'wb') as f:
    f.write(b)

  print(f"✅ 녹음 완료! 파일이 '{filename}'로 저장되었습니다.")


In [10]:
# # =========================
# # 2) 녹음된 파일로 실행
# # =========================
# audio_path = my_path / "data" / "dataset" / "wav_jjy" / "raw" / "audio_004.m4a"

# # 1. STT
# stt_result = whisper_model.transcribe(str(audio_path))
# question = stt_result["text"].strip()
# print("STT 결과:", question)

# # 2. LLM
# answer = chat_qwen(question)
# print("LLM 답변:", answer)

# # 3. TTS
# wavs, sr = tts_model.generate_custom_voice(
#     text=answer,
#     speaker=speaker,
#     # instruct="very angry"
# )

# for i, wav in enumerate(wavs):
#     out_path = f"/content/{speaker}_result_{i+1}.wav"
#     sf.write(out_path, wav, sr)

#     print(f"\n===== 문장 {i+1} =====")
#     print(answer)
#     display(Audio(out_path))

In [11]:
# # =========================
# # 2) 녹음 후 실행
# # =========================
# from IPython.display import display, Javascript
# from google.colab import output
# from base64 import b64decode

# # 1. 브라우저 마이크 접근을 위한 JavaScript 코드
# RECORD_JS = """
# const sleep  = time => new Promise(resolve => setTimeout(resolve, time));
# const b2text = blob => new Promise(resolve => {
#   const reader = new FileReader();
#   reader.onloadend = e => resolve(e.srcElement.result);
#   reader.readAsDataURL(blob);
# });

# var record = time => new Promise(async resolve => {
#   stream = await navigator.mediaDevices.getUserMedia({ audio: true });
#   recorder = new MediaRecorder(stream);
#   chunks = [];
#   recorder.ondataavailable = e => chunks.push(e.data);
#   recorder.start();
#   await sleep(time);
#   recorder.onstop = async ()=>{
#     blob = new Blob(chunks);
#     text = await b2text(blob);
#     resolve(text);
#   };
#   recorder.stop();
# });
# """

# # 2. 실행 (원하는 초 설정)
# record_audio(sec=10)

# audio_path = "/content/recorded_audio.wav"

# # 1. STT
# stt_result = whisper_model.transcribe(str(audio_path))
# question = stt_result["text"].strip()
# print("STT 결과:", question)

# # 2. LLM
# answer = chat_qwen(question)
# print("LLM 답변:", answer)

# # 3. TTS
# wavs, sr = tts_model.generate_custom_voice(
#     text=answer,
#     speaker=speaker,
#     # instruct="very angry"
# )

# for i, wav in enumerate(wavs):
#     out_path = f"/content/{speaker}_result_{i+1}.wav"
#     sf.write(out_path, wav, sr)

#     print(f"\n===== 문장 {i+1} =====")
#     print(answer)
#     display(Audio(out_path))

In [12]:
# =========================
# 2) gradio를 통해 실행
# =========================
import gradio as gr
import soundfile as sf
from IPython.display import Audio

def run_pipeline(audio_path):
    # 1) STT
    stt_result = whisper_model.transcribe(audio_path)
    question = stt_result["text"].strip()

    # 2) LLM
    answer = chat_qwen(question)

    # 3) TTS
    wavs, sr = tts_model.generate_custom_voice(
        text=answer,
        speaker=speaker
    )

    out_path = "/content/gradio_reply.wav"
    sf.write(out_path, wavs[0], sr)

    return question, answer, out_path

with gr.Blocks() as demo:
    gr.Markdown("## 음성 대화 테스트")

    audio_in = gr.Audio(
        sources=["microphone"],
        type="filepath",
        format="wav",
        label="마이크 입력"
    )
    run_btn = gr.Button("실행")
    stt_box = gr.Textbox(label="STT 결과")
    llm_box = gr.Textbox(label="LLM 답변")
    tts_audio = gr.Audio(label="TTS 출력", type="filepath")

    run_btn.click(
        fn=run_pipeline,
        inputs=audio_in,
        outputs=[stt_box, llm_box, tts_audio]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9474596bc3f56b7ee8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
